In [0]:
dim_appointment_type = spark.sql(f"select * from regis_healthcare.silver.appointments;")
dim_appointment_type.createOrReplaceTempView("appointments")

In [0]:
# 8. Dim_Appointment
# | Column               |
# | -------------------- |
# | appointment_type_key |
# | appointment_type     |

dim_appointment_type = spark.sql("""select * from appointments""")
# display(dim_appointment_type)
#----------------
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# dim_appointment_type = dim_appointment_type.select(
#     "appointment_type"
# ).distinct()
# window_spec = Window.orderBy("appointment_type")
dim_appointment_type = spark.sql("""
    select distinct(appointment_type) from appointments """)
dim_appointment_type.createOrReplaceTempView("aps")
# dim_appointment_type = dim_appointment_type.withColumn(
#     "appointment_type_key",
#     row_number().over(window_spec)
# )
dim_appointment_type = spark.sql("""
    select appointment_type,row_number()over(order by appointment_type )as appointment_type_key from aps group by appointment_type """)
#----------------
dim_appointment_type = dim_appointment_type.select(
"appointment_type_key",
"appointment_type")
display(dim_appointment_type)

#### cataloge 

In [0]:
# dim_appointment_type.write\
#     .format("delta")\
#     .option("mergeSchema","true")\
#     .option("overwriteSchema","true")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .saveAsTable(f"regis_healthcare.gold.dim_appointment_type")

In [0]:
dim_appointment_type.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.sb_dim_appointment_type")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.dim_appointment_type")
# Create DataFrame from source table with correct fully-qualified name
# sb_dim_products
df_child_products = (
    spark.table("regis_healthcare.gold.sb_dim_appointment_type")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.appointment_type_key = source.appointment_type_key"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from regis_healthcare.gold.dim_appointment_type;")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from regis_healthcare.gold.sb_dim_appointment_type;")
print(sb_dim_df.count())

#### s3 loading

In [0]:
# # gold load to s3
# dim_appointment_type.write\
#     .format("delta")\
#     .option("mergeSchema","true")\
#     .option("overwriteSchema","true")\
#         .option("delta.enableChangeDataFeed","true")\
#             .mode("overwrite")\
# .save(f"s3://regis-healthcare/gold-delta-table/dim_appointment_type")

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/dim_appointment_type"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = dim_appointment_type

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.appointment_type_key = source.appointment_type_key"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)
